In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
groqAPI = os.getenv("groqAPI")
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=groqAPI
)

: 

In [ ]:
from typing import TypedDict, Dict, Any

class DispatchState(TypedDict):
    effort_vectors: Dict
    drivers: Dict
    plan: Dict
    allocation: Dict
    critique: Dict
    explanation: Dict

: 

In [3]:
from optimized_allocation import allocateDrivers_optimized
import json

def allocateDrivers(effortVectors, driverData):
    return allocateDrivers_optimized(effortVectors, driverData)

def openJson(filePath):
    with open(filePath, 'r') as file:
        return json.load(file)


In [4]:
def planner_agent(state: DispatchState):

    effort_vectors = state["effort_vectors"]

    # simple rule-based (can upgrade to LLM later)
    plan = {
        "strategy": "fairness_strict",
        "num_clusters": len(effort_vectors),
        "notes": "Prioritize fairness over efficiency"
    }

    return {"plan": plan}

In [5]:
def allocation_agent(state: DispatchState):

    effort_vectors = state["effort_vectors"]
    drivers = state["drivers"]

    allocation = allocateDrivers(effort_vectors, drivers)

    return {"allocation": allocation, "drivers": drivers}

In [6]:
def critic_agent(state: DispatchState):

    allocation = state["allocation"]

    prompt = f"""
You are an expert in logistics fairness systems.

Analyze this allocation:
{allocation}

IMPORTANT:
- Return ONLY valid JSON
- Do NOT include explanations
- Do NOT include <think> or reasoning

Strict format:
{{
  "score": 0.0,
  "issues": [],
  "suggestion": ""
}}
"""

    response = llm.invoke(prompt)

    import json
    try:
        critique_json = json.loads(response.content)
    except:
        critique_json = {
            "score": 0.5,
            "issues": ["invalid_json_response"],
            "suggestion": "LLM output not structured properly"
        }

    return {"critique": critique_json}

In [7]:
import json

def reallocation_agent(state: DispatchState):

    critique = state["critique"]
    score = critique.get("score", 1)

    if score > 0.75:
        return {}
    print("⚠️ Reallocation triggered")
    for d in state["drivers"].values():
        d["consecutive_heavy_days"] = max(0, d["consecutive_heavy_days"] - 1)

    new_allocation = allocateDrivers(
        state["effort_vectors"],
        state["drivers"]
    )

    return {"allocation": new_allocation}

In [ ]:
def explainer_agent(state: DispatchState):

    allocation = state["allocation"]

    prompt = f"""
    Explain this driver allocation in simple terms.

    {allocation}

    For each cluster:
    - Why that driver was chosen
    - Mention fairness and constraints

    Keep it clear and concise.
    """

    response = llm.invoke(prompt)

    return {"explanation": response.content}

In [9]:
from langgraph.graph import StateGraph, END

builder = StateGraph(DispatchState)

builder.add_node("planner", planner_agent)
builder.add_node("allocator", allocation_agent)
builder.add_node("critic", critic_agent)
builder.add_node("reallocator", reallocation_agent)
builder.add_node("explainer", explainer_agent)

In [10]:
def should_reallocate(state: DispatchState):

    critique = state["critique"]

    return critique.get("score", 1) < 0.60

In [11]:
builder.set_entry_point("planner")

builder.add_edge("planner", "allocator")
builder.add_edge("allocator", "critic")

builder.add_conditional_edges(
    "critic",
    should_reallocate,
    {
        True: "reallocator",
        False: "explainer"
    }
)

builder.add_edge("reallocator", "explainer")
builder.add_edge("explainer", END)

graph = builder.compile()

In [12]:
from IPython.display import display, Markdown

mermaid_code = graph.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	allocator(allocator)
	critic(critic)
	reallocator(reallocator)
	explainer(explainer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	allocator --> critic;
	critic -. &nbsp;False&nbsp; .-> explainer;
	critic -. &nbsp;True&nbsp; .-> reallocator;
	planner --> allocator;
	reallocator --> explainer;
	explainer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [13]:
import json

with open("jsonFiles/finalFeatures.json", "r") as f:
    effortVectors = json.load(f)

with open("jsonFiles/driversdata.json", "r") as f:
    driverData = json.load(f)

In [ ]:

result = graph.invoke({
    "effort_vectors": effortVectors,
    "drivers": driverData
})

Optimized Allocation Complete. Time: 0.2627s


In [15]:
print(result["allocation"])
print(result["critique"])
print(result["explanation"])

{'Cluster 4': 'D18', 'Cluster 5': 'D17', 'Cluster 1': 'D40', 'Cluster 0': 'D37', 'Cluster 3': 'D41', 'Cluster 2': 'D38'}
{'score': 1.0, 'issues': [], 'suggestion': ''}
**Driver‑to‑Cluster Assignment – What’s going on?**  

| Cluster | Driver | Why this driver fits the cluster | Fairness & Constraints kept in mind |
|--------|--------|-----------------------------------|--------------------------------------|
| **Cluster 0** | **D37** | • Lives/works closest to the geographic area of Cluster 0, so travel time is minimal.<br>• Has the vehicle type required for the jobs in this cluster (e.g., a midsize van). | • D37 has only 1 shift left this week, so giving the cluster now balances his workload with other drivers.<br>• No other driver is within the 10‑km radius, so the choice is both fair and necessary. |
| **Cluster 1** | **D40** | • Holds the special certification (e.g., hazardous‑materials) that many orders in Cluster 1 need.<br>• Has a strong on‑time record for the high‑volume routes

In [7]:
import requests

result = requests.post("http://localhost:8000/dispatch/5")

In [8]:
print(result.json())

{'allocation': {'Cluster 8': 'D75', 'Cluster 5': 'D30', 'Cluster 1': 'D6', 'Cluster 3': 'D22', 'Cluster 2': 'D75', 'Cluster 4': 'D11', 'Cluster 0': 'D17', 'Cluster 7': 'D11', 'Cluster 9': 'D6', 'Cluster 6': 'D60'}, 'fairness_report': {'driver_workloads': {'D75': 562144.786, 'D30': 287883.661, 'D6': 518890.924, 'D22': 265825.715, 'D11': 531725.889, 'D17': 250151.29, 'D60': 235641.358}, 'fairness_score': 0.6339, 'strategy_used': 'balanced', 'swaps_applied': 0}, 'critique': {'score': 0.55, 'issues': ['Workload distribution is highly uneven (standard deviation ~35% of mean).', 'Anomaly clusters 2 and 8 are assigned to D75, who already has a consecutive_heavy count of 10, indicating insufficient rest.', "D75's total workload (562k) is the highest, further compounding strain on an already over‑burdened driver."], 'suggestion': 'Move Cluster 8 (or Cluster 2) from D75 to a driver with lower workload and fewer consecutive heavy days, such as D30 or D60, to balance both workload and rest require